# 05 — Solving Quantum OT in cvxpy

Point the coupling SDP at the example the module has carried since the start — separate $|+\rangle\langle+|$ from $I/2$, read the optimal coupling, and close the transport row of the running dictionary.

**Prerequisites:** `04/04_quantum_coupling_sdp`, `03/03_the_kantorovich_lp`.

**What you'll be able to do**
- Confirm the **diagonal-collapse consistency check** numerically: for commuting states the quantum SDP returns the *classical* $W_2^2$ on their diagonals.
- Deliver the **$|+\rangle\langle+|$ vs $I/2$ punchline** — classical OT on the diagonals returns zero, the QOT SDP returns a strictly positive value.
- **Read an optimal coupling** $\rho_{AB}^\star$: solve, recover the two marginals by partial trace, and see the operator-level transport plan.
- Add the explicit **coupling-SDP row** to the classical↔quantum dictionary.

In `04/04` you built the machinery: you defined a quantum coupling, lifted the Kantorovich LP to a semidefinite program, solved it in a few lines of cvxpy, and validated it against a closed form. The tool is trustworthy. Now we point it at the example that has motivated the whole module — the two qubit states that share a $Z$-diagonal yet are physically different. Classical optimal transport, working from those diagonals alone, calls them identical. We watch the SDP disagree, look at the coupling it returns, and write the row this arc has been earning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.quantum.composite import partial_trace
from qot_course.quantum.density import density_matrix, maximally_mixed
from qot_course.quantum.states import KET_PLUS
from qot_course.quantum_ot.sdp import (
    swap_cost,
    quadratic_position_cost,
    quantum_ot_sdp,
)

np.random.seed(42)
viz.use_course_style()

## The consistency check — commuting states give the classical answer

Before we let the SDP tell us something *new*, we check it still agrees with everything we already know. The cleanest test is the **diagonal-collapse principle** of `04/01`: when $\rho_A$ and $\rho_B$ **commute** — both diagonal in the same basis — they carry no coherence the classical picture misses, so the quantum OT SDP must return the *same number* as the classical Kantorovich LP run on their diagonals.

We make this concrete with the quadratic position cost. The operator $C = (X_A \otimes I - I \otimes X_B)^2$, with $X = \mathrm{diag}(\text{positions})$, is the operator lift of the classical ground cost $C_{ij} = (i - j)^2$ — it charges transport by squared distance along the position axis (`04/04` met its sibling, the SWAP cost; this is the cost that recovers the classical $W_2^2$). For two diagonal states in the $X$ eigenbasis, the SDP and the LP are solving the *same* problem written in two languages. We take $\rho_A = \mathrm{diag}(0.6, 0.4)$ and $\rho_B = \mathrm{diag}(0.1, 0.9)$ on positions $\{0, 1\}$, and compare.

In [ ]:
# Two diagonal density matrices in the Z basis (they commute).
p_vec = np.array([0.6, 0.4])
q_vec = np.array([0.1, 0.9])
rho_a_diag = np.diag(p_vec).astype(complex)
rho_b_diag = np.diag(q_vec).astype(complex)

# Quantum SDP with the quadratic position cost (positions = {0, 1}).
positions = np.array([0.0, 1.0])
qot_value, _ = quantum_ot_sdp(
    rho_a_diag, rho_b_diag, quadratic_position_cost(positions)
)

# Classical W_2^2 with cost C[i, j] = (i - j)^2 on the same diagonals.
classical_cost = (positions.reshape(-1, 1) - positions.reshape(1, -1)) ** 2
w2_sq_classical = float(ot.emd2(p_vec, q_vec, classical_cost))

print(f"Quantum SDP value (quadratic position cost) = {qot_value:.4f}")
print(f"Classical W_2^2  (LP on the Z-diagonals)    = {w2_sq_classical:.4f}")
print(f"diagonal-collapse match?                      {np.isclose(qot_value, w2_sq_classical, atol=1e-5)}")

**Read the output.** The two numbers agree to four decimals. The quantum SDP, handed a pair of commuting states, returns exactly the classical Kantorovich answer on their diagonals — Trevisan's consistency principle, confirmed by hand. This is the reason we *trust* the SDP: it reproduces everything the classical world already knows, and so any place where it returns something different must be genuinely new physics, not a modelling artefact. That is the door we open next.

## The punchline — the SDP separates $|+\rangle\langle+|$ from $I/2$

Here is the example the module has pointed at since `04/01`. Take the pure superposition $|+\rangle\langle+|$ and the maximally mixed state $I/2$. Both have the **same $Z$-diagonal**, $(\tfrac12, \tfrac12)$: measure either one in the computational basis and you get an even coin. Classical optimal transport sees only those diagonals, so it judges the two states *identical* and returns a transport cost of zero. Yet they could not be more different — one is a pure state with full coherence, the other is the most mixed state there is.

The SDP does not work from the diagonals; it works from the operators. We run it on this pair with both cost operators and watch it return a **strictly positive** value where the classical LP returned zero.

In [ ]:
plus = density_matrix(KET_PLUS)
mixed = maximally_mixed(2)

# Both have Z-diagonal (1/2, 1/2); classical OT on the diagonals returns zero.
diag_plus = np.diag(plus).real.copy()
diag_mixed = np.diag(mixed).real.copy()
classical_value = float(ot.emd2(
    np.ascontiguousarray(diag_plus),
    np.ascontiguousarray(diag_mixed),
    classical_cost,
))

# The SDP works from the operators, not the diagonals — and sees the difference.
qot_swap_value, _ = quantum_ot_sdp(plus, mixed, swap_cost(2))
qot_quad_value, _ = quantum_ot_sdp(plus, mixed, quadratic_position_cost(positions))

print(f"Classical W_2^2 on the Z-diagonals (both (1/2, 1/2)) = {classical_value:.4f}   <- cannot tell them apart")
print(f"Quantum SDP, SWAP cost                                = {qot_swap_value:.4f}")
print(f"Quantum SDP, quadratic position cost                  = {qot_quad_value:.4f}")
print()
print(f"both strictly positive?  {qot_swap_value > 1e-3 and qot_quad_value > 1e-3}")

**Read the output.** Classical optimal transport on the diagonals returns zero — from where it stands, $|+\rangle\langle+|$ and $I/2$ are the same distribution. The quantum SDP returns a clearly positive value under *both* cost operators (here $0.5$ for each). The SDP sees what the diagonal threw away: the off-diagonal coherence that distinguishes a pure superposition from a uniform mixture.

This is the formal payoff for the whole arc. We now hold a single object — the coupling SDP — that **agrees** with classical OT whenever the states commute (the check above) and **strictly separates** states that share a diagonal but differ in their coherences. It does not contradict the classical theory; it extends it into exactly the region where the classical theory was structurally blind.

## See the optimal coupling

The SDP returns more than a number. Its optimal variable is the full **optimal coupling** $\rho_{AB}^\star$ — for two qubits, a $4 \times 4$ Hermitian PSD matrix, the operator-level analogue of the transport plan we drew as mass-weighted arrows in module 03. We solve once more for the $|+\rangle\langle+|$ vs $I/2$ case, keep that second return value, confirm it is an honest coupling by recovering both marginals through partial trace, and then look at it.

In [ ]:
qot_value, optimal_plan = quantum_ot_sdp(plus, mixed, quadratic_position_cost(positions))

# An honest coupling: tracing out each subsystem must return the prescribed marginal.
rho_a_back = partial_trace(optimal_plan, keep=[0], dims=[2, 2])
rho_b_back = partial_trace(optimal_plan, keep=[1], dims=[2, 2])
print("marginal A recovered (should equal |+><+|):")
print(np.round(rho_a_back, 4))
print()
print("marginal B recovered (should equal I/2):")
print(np.round(rho_b_back, 4))
print()
print(f"trace of plan (should be 1)             = {np.real(np.trace(optimal_plan)):.4f}")
# PSD => the smallest eigenvalue is zero up to interior-point tolerance (~1e-9).
min_eig = float(np.min(np.linalg.eigvalsh(0.5 * (optimal_plan + optimal_plan.conj().T))))
print(f"smallest eigenvalue (PSD => >= 0)       = {min_eig:.2e}")

# Coupling on H_A (x) H_B, both qubits -> the two-qubit tensor basis.
two_qubit_basis = ["|00>", "|01>", "|10>", "|11>"]
viz.plot_density_matrix(
    optimal_plan,
    title=r"Optimal coupling $\rho^\star_{AB}$ for $|+\rangle\langle+|$  vs  $I/2$",
    basis_labels=two_qubit_basis,
)
plt.show()

**Read the figure and the output.** The recovered marginals land exactly where they must: tracing out $B$ returns $|+\rangle\langle+|$ (every entry $\tfrac12$), tracing out $A$ returns $I/2$ (a half on the diagonal, zero off it). The plan has unit trace, and its smallest eigenvalue is zero to solver tolerance — a number on the order of $10^{-9}$, the interior-point residual, confirming $\rho_{AB}^\star \succeq 0$.

The figure shows the coupling as two heatmaps side by side — its real part on the left, its imaginary part on the right. The real part is where the action is: alongside the diagonal weight there are off-diagonal entries that tie the $A$ and $B$ registers together, structure that carries the coherence of $|+\rangle\langle+|$ into the joint state. The imaginary part is flat — every entry zero — because this particular optimum is real; a coupling between states with complex coherences would light it up. This is the operator-level counterpart of the arrow diagrams from module 03: there the "flow" was mass moving between positions, here it is operator correlation binding two subsystems, and *no classical coupling* — a non-negative matrix on the diagonals — could reproduce it.

## The QOT dictionary row

Module 03 opened the transport row of the classical↔quantum dictionary; `04/03` and `04/04` filled in the coupling and the SDP. With the consistency check and the punchline both in hand, we can now write the row in full — including the entry that makes the quantum side worth the trouble: the SDP separates states the classical LP cannot.

| Classical | Quantum |
|-----------|---------|
| transport plan / coupling $P \in T(a, b)$ | quantum coupling $\rho_{AB} \succeq 0$ with $\mathrm{tr}_B\,\rho_{AB} = \rho_A,\ \mathrm{tr}_A\,\rho_{AB} = \rho_B$ |
| Kantorovich LP $\min \langle C, P\rangle$ | quantum OT SDP $\min \mathrm{tr}(C\,\rho_{AB})$ |
| transportation polytope $T(a, b)$ | set of quantum couplings (a *spectrahedron*) |
| ground cost matrix $C_{ij}$ | cost operator $C$ (SWAP cost; quadratic position cost) |
| $W_2^2$ on commuting states | **same $W_2^2$** — diagonal-collapse consistency |
| blind to coherence: $|+\rangle\langle+|$ and $I/2$ look identical | **strictly separates them**: a positive transport cost |

The last two rows are the whole story of this arc. On commuting states the quantum theory *is* the classical theory; off them, it sees the coherence the diagonal discards.

## Your turn

**Warm-up.** Re-run the punchline with a *different* pure state that still has the even $Z$-diagonal — for instance $|{+}i\rangle = (|0\rangle + i|1\rangle)/\sqrt{2}$, built with `density_matrix([1, 1j])`. Its diagonal is again $(\tfrac12, \tfrac12)$, so classical OT still returns zero. Predict whether the SDP value against $I/2$ will be positive, then solve and check. What does that tell you about how much of the difference the diagonal was hiding?

**Go further.** Swap the cost operator and watch the consistency check from the other side. Take the same two *commuting* diagonal states from the first section and solve the SDP with the **SWAP** cost instead of the quadratic position cost. The SWAP cost charges by disagreement between subsystems rather than by squared position distance, so it need not return the classical $(i-j)^2$ value — but it should still vanish when the states are identical. Verify both: that identical commuting states cost zero under SWAP, and that two *different* commuting states need not match the quadratic-cost number. What does this say about the cost operator as a modelling choice?

**Challenge.** Make the separation a *curve*, not a single point. Interpolate between the pure state and the mixed state along $\rho(t) = (1-t)\,|+\rangle\langle+| + t\,(I/2)$ for $t \in [0, 1]$ — every $\rho(t)$ keeps the even $Z$-diagonal, so classical OT returns zero across the whole sweep. Solve the SDP for $\rho(t)$ against $I/2$ at several values of $t$ and describe the curve of QOT values: where is it largest, where does it reach zero, and why does that endpoint behaviour make physical sense?

## What you built

- You confirmed the **diagonal-collapse consistency check**: for commuting states the quantum SDP returns the classical $W_2^2$ on their diagonals, exactly — the principle that lets you trust the solver.
- You delivered the **$|+\rangle\langle+|$ vs $I/2$ punchline** the module has carried since `04/01`: classical OT on the shared diagonal returns zero, the QOT SDP returns a strictly positive cost. The SDP sees the coherence the diagonal discards.
- You **read an optimal coupling** $\rho_{AB}^\star$: solved for it, recovered both marginals by partial trace, checked it is PSD with unit trace, and saw the operator correlations no classical coupling can express.
- You wrote the explicit **coupling-SDP row** of the classical↔quantum dictionary.

Take a moment with this: the question that opened the module — *can transport tell apart two states that look the same on the diagonal?* — now has a worked, visualised answer. Yes, and here is the object that does it. The dictionary row you have added here is the record of that promise kept:

| Classical | Quantum |
|-----------|---------|
| Kantorovich LP $\min \langle C, P\rangle$ over couplings $P \in T(a, b)$ | quantum OT SDP $\min \mathrm{tr}(C\,\rho_{AB})$ over couplings $\rho_{AB} \succeq 0$ |
| agrees with $W_2^2$ on commuting states | **strictly separates** $|+\rangle\langle+|$ from $I/2$ — coherence the diagonal hid |

## What's next

This SDP returns a single optimal coupling — the sharp, sometimes brittle minimiser. In `06_entropic_qot_gibbs` we add a von Neumann entropy term to smooth it, and find the matrix-exponential Gibbs kernel waiting at the centre — the operator analogue of the entropic regularisation of module 03.

## References

- G. De Palma & D. Trevisan, "Quantum optimal transport with quantum channels", *Annales Henri Poincaré* 22, 3199–3234 (2021). DOI:10.1007/s00023-021-01042-3
- S. Cole, M. Lostaglio, K. Verma & M. M. Wilde, "Quantum Wasserstein distance based on an optimization over separable states", *IEEE Transactions on Information Theory* (2023). DOI:10.1109/TIT.2023.3328953
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091
- S. P. Boyd & L. Vandenberghe, *Convex Optimization*, Cambridge University Press (2004), chs. 4 & 11 (semidefinite programming).

**Previous:** `notebooks/04_QuantumOptimalTransport/04_quantum_coupling_sdp.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/06_entropic_qot_gibbs.ipynb`